# PlannerAgent GCC-4J — Remote GPU execution
Select a GPU runtime, upload the prepared INPUT zip, then Run all.

In [ ]:
import os, platform, shutil, subprocess, sys
subprocess.run(['nvidia-smi'], check=True)
print(platform.platform(), platform.processor(), shutil.disk_usage('/content'))

In [ ]:
!pip -q install 'transformers>=4.51.0' datasets peft accelerate safetensors psutil
os.environ['WANDB_DISABLED']='true'
os.environ['HF_HUB_DISABLE_TELEMETRY']='1'

In [ ]:
from google.colab import files
uploaded = files.upload()
archives = [name for name in uploaded if name.endswith('.zip')]
assert len(archives) == 1, 'Upload exactly one GCC4J INPUT zip'
shutil.unpack_archive(archives[0], '/content/gcc4j')
print('Bundle extracted')

In [ ]:
import hashlib, pathlib
root=pathlib.Path('/content/gcc4j')
sums=(root/'SHA256SUMS.txt').read_text().splitlines()
for line in sums:
    expected, relative=line.split('  ',1); actual=hashlib.sha256((root/relative).read_bytes()).hexdigest(); assert actual==expected, relative
print('Input bundle hashes: PASS')

In [ ]:
!cd /content/gcc4j/scripts && python train_first_local_student.py

In [ ]:
result='/content/gcc4j/PA-INTERPRETATION-STUDENT-v0.1-GCC4J.zip'
assert os.path.exists(result)
print(result, os.path.getsize(result), hashlib.sha256(open(result,'rb').read()).hexdigest())
files.download(result)